In [2]:
# 04_rcfs.ipynb

# %% [markdown]
# # Regulatory-Constrained Fairness Framework for Credit Scoring
# ## 04. Regulatory-Constrained Fairness Score (RCFS)
#
# RCFS is a compliance score (higher = better) that fuses model performance with a
# harmonic combination of regulatory-requirement satisfaction scores.
#
# Component satisfaction scores (each in [0,1], higher = better):
#   c_perf : discriminative adequacy   = max(0, (AUC - 0.5) / 0.5)
#   c_fair : fairness                  = 0.5*min(1, DI) + 0.5*(1 - |SPD|)   (worst-case)
#   c_expl : explainability            = 1 - normalized entropy of |SHAP|
#   c_rec  : recourse feasibility      = fraction of adverse predictions with a
#                                        feasible counterfactual (evaluation only)
#
# Compliance is the weighted HARMONIC mean of {c_fair, c_expl, c_rec}, so a near-zero on
# any single requirement collapses the score ("no requirement may be sacrificed"):
#   Compliance = (sum w_k) / (sum w_k / c_k)
# RCFS balances performance vs compliance through gamma:
#   RCFS(gamma) = (1 - gamma) * c_perf + gamma * Compliance
#
# This notebook compares fairness-mitigation techniques (pre / in / post-processing) by
# the RCFS they achieve, and sweeps gamma to trace the performance-compliance curve.
# A single-regulation, threshold-free continuous score is used (regulation comparison is
# out of scope here); recourse enters the evaluation RCFS but not the training loss.

# %%
# Imports and configuration
import os
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import shap

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")

RANDOM_STATE = 42
SHAP_SAMPLE = 1000
os.makedirs(TAB_DIR, exist_ok=True)

# RCFS component weights (for the harmonic compliance term) and default gamma
W_FAIR, W_EXPL, W_REC = 1.0, 1.0, 1.0
GAMMA_DEFAULT = 0.5
GAMMA_SWEEP = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]

DATASETS = [
    ("German Credit",        "german",               "sex", "male"),
    ("Taiwan CC Default",    "taiwan",                "sex", "male"),
    ("Credit Card Approval", "credit_card_approval",  "sex", "male"),
]

# %%
# Helpers
with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json")) as f:
    META = json.load(f)

def load_split(key):
    train = pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_train.parquet"))
    test = pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_test.parquet"))
    return train, test

def get_Xy(df, key):
    meta = META[key]
    target_col = meta["target_col"]
    sensitive_cols = meta["sensitive_cols"]
    aux_cols = [c for c in ["target_60dpd", "target_30dpd"] if c in df.columns and c != target_col]
    drop = [target_col] + sensitive_cols + aux_cols
    X = df.drop(columns=[c for c in drop if c in df.columns], errors="ignore")
    y = df[target_col].astype(int)
    return X, y

def di_spd(y_pred, priv_mask, unpriv_mask):
    ppr_p = y_pred[priv_mask].mean() if priv_mask.sum() else np.nan
    ppr_u = y_pred[unpriv_mask].mean() if unpriv_mask.sum() else np.nan
    if not (np.isfinite(ppr_p) and np.isfinite(ppr_u)) or ppr_p == 0:
        return np.nan, np.nan
    return ppr_u / ppr_p, ppr_u - ppr_p

# Fairness satisfaction on the primary sensitive attribute (privileged = given value)
def c_fair_score(test_df, sens_col, privileged, y_pred):
    s = test_df[sens_col]
    priv_mask = (s == privileged).values
    unpriv_mask = (s != privileged).values & s.notna().values
    di, spd = di_spd(y_pred, priv_mask, unpriv_mask)
    if not np.isfinite(di):
        return np.nan
    return 0.5 * min(1.0, di) + 0.5 * (1.0 - abs(spd))

# %%
# Explainability satisfaction score (rescaled).
# Problem with 1 - normalized_entropy: it is structurally low for models that spread
# small attributions across many features (e.g. XGBoost with wide feature sets),
# because the reference is a uniform distribution over ALL features.
#
# Rescaled definition using the effective number of features:
#   entropy  = -sum p log p         over the mean |SHAP| distribution
#   n_eff    = exp(entropy)          effective number of contributing features
#   c_expl   = min(1, k_star / n_eff)
# Rationale: regulatory explainability needs a small set of reason codes; if the model
# effectively relies on <= k_star features it is "explainable enough" (score 1), and the
# score degrades proportionally as more features are effectively involved. This is
# invariant to the total feature count, so it is scale-comparable with c_fair.
EXPL_KSTAR = 10  # target number of effective reason-code features

def c_expl_score(model, X, sample=SHAP_SAMPLE, k_star=EXPL_KSTAR, seed=RANDOM_STATE):
    if len(X) > sample:
        X = X.sample(sample, random_state=seed)
    try:
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X)
    except Exception:
        explainer = shap.Explainer(model, X)
        sv = explainer(X).values
    if isinstance(sv, list):
        sv = sv[-1]
    mean_abs = np.abs(sv).mean(axis=0)
    total = mean_abs.sum()
    if total == 0:
        return 0.0
    p = mean_abs / total
    p = p[p > 0]
    if len(p) <= 1:
        return 1.0
    entropy = -np.sum(p * np.log(p))
    n_eff = np.exp(entropy)                 # effective number of features
    return float(min(1.0, k_star / n_eff))

def prevalence_threshold(proba, y_train):
    return np.quantile(proba, 1.0 - float(y_train.mean()))

# %%
# Recourse feasibility (evaluation only). Reuses the DiCE approach from the cross-domain
# study but is optional: if dice-ml is unavailable, c_rec falls back to NaN and is dropped
# from the harmonic mean (documented as a limitation).
def c_rec_score(model_pipeline_available=False):
    # Placeholder hook: recourse is computed in a dedicated routine below only if enabled.
    return np.nan

# Toggle recourse computation (kept off by default to keep this notebook self-contained;
# the cross-domain recourse notebook demonstrates the full DiCE pipeline).
COMPUTE_RECOURSE = False

# %%
# Fairness-mitigation techniques.
# We implement portable versions that do not require AIF360:
#   - baseline            : class-weighted XGBoost (no mitigation)
#   - reweighing (pre)     : sample weights equalizing P(y) across groups
#   - reject_option (post) : group-specific decision thresholds equalizing positive rates
# Adversarial debiasing (in-processing) is noted as an extension requiring AIF360.
def group_reweighing_weights(y, s):
    # Kamiran-Calders style reweighing: w = P(y)P(s) / P(y,s)
    w = np.ones(len(y), dtype=float)
    n = len(y)
    for yy in np.unique(y):
        for ss in pd.unique(s):
            mask = (y == yy) & (s.values == ss)
            n_ys = mask.sum()
            if n_ys == 0:
                continue
            p_y = (y == yy).mean()
            p_s = (s.values == ss).mean()
            p_ys = n_ys / n
            w[mask] = (p_y * p_s) / p_ys
    return w

def reject_option_thresholds(proba, s, y_train_rate, privileged):
    # Post-processing: choose per-group thresholds so each group's positive rate matches
    # the overall prevalence (a simple statistical-parity-oriented adjustment).
    thr = {}
    for grp in pd.unique(s):
        p = proba[(s.values == grp)]
        if len(p) == 0:
            thr[grp] = 0.5
        else:
            thr[grp] = np.quantile(p, 1.0 - y_train_rate)
    return thr

# %%
# Train one technique on one dataset, return predictions + fitted model for scoring
def run_technique(technique, X_train, y_train, X_test, s_train, s_test, privileged):
    n_pos = int(y_train.sum()); n_neg = int((y_train == 0).sum())
    spw = (n_neg / n_pos) if n_pos > 0 else 1.0
    base = dict(n_estimators=300, max_depth=5, learning_rate=0.05,
                subsample=0.9, colsample_bytree=0.9,
                eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)

    if technique == "baseline":
        model = XGBClassifier(scale_pos_weight=spw, **base)
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        thr = prevalence_threshold(proba, y_train)
        y_pred = (proba >= thr).astype(int)

    elif technique == "reweighing":
        w = group_reweighing_weights(y_train.values, s_train)
        model = XGBClassifier(scale_pos_weight=spw, **base)
        model.fit(X_train, y_train, sample_weight=w)
        proba = model.predict_proba(X_test)[:, 1]
        thr = prevalence_threshold(proba, y_train)
        y_pred = (proba >= thr).astype(int)

    elif technique == "reject_option":
        model = XGBClassifier(scale_pos_weight=spw, **base)
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        thr_map = reject_option_thresholds(proba, s_test, float(y_train.mean()), privileged)
        y_pred = np.array([
            int(p >= thr_map.get(g, 0.5)) for p, g in zip(proba, s_test.values)
        ])

    else:
        raise ValueError(technique)

    return model, proba, y_pred

# %%
# Compute RCFS for every (dataset x technique)
TECHNIQUES = ["baseline", "reweighing", "reject_option"]

def compliance_harmonic(cs, ws):
    pairs = [(c, w) for c, w in zip(cs, ws) if np.isfinite(c) and c > 0]
    if not pairs:
        return np.nan
    num = sum(w for _, w in pairs)
    den = sum(w / c for c, w in pairs)
    return num / den if den > 0 else np.nan

def rcfs(c_perf, compliance, gamma):
    if not np.isfinite(compliance):
        return np.nan
    return (1 - gamma) * c_perf + gamma * compliance

rows = []
detail = {}

for display_name, key, sens_col, privileged in DATASETS:
    train, test = load_split(key)
    X_train, y_train = get_Xy(train, key)
    X_test, y_test = get_Xy(test, key)
    X_test = X_test[X_train.columns]
    s_train = train[sens_col]
    s_test = test[sens_col]

    for tech in TECHNIQUES:
        model, proba, y_pred = run_technique(
            tech, X_train, y_train, X_test, s_train, s_test, privileged
        )
        auc = roc_auc_score(y_test.values, proba)
        c_perf = max(0.0, (auc - 0.5) / 0.5)
        c_fair = c_fair_score(test, sens_col, privileged, y_pred)
        c_expl = c_expl_score(model, X_test)
        c_rec = c_rec_score(COMPUTE_RECOURSE)  # NaN unless enabled

        compliance = compliance_harmonic([c_fair, c_expl, c_rec], [W_FAIR, W_EXPL, W_REC])
        score = rcfs(c_perf, compliance, GAMMA_DEFAULT)

        rows.append({
            "dataset": display_name, "technique": tech,
            "AUC": auc, "c_perf": c_perf, "c_fair": c_fair,
            "c_expl": c_expl, "c_rec": c_rec,
            "Compliance": compliance, "RCFS": score,
        })
        detail[(display_name, tech)] = {"c_perf": c_perf, "compliance": compliance}

        print(f"[{display_name:>22}] {tech:<14} "
              f"AUC={auc:.3f}  c_fair={c_fair:.3f}  c_expl={c_expl:.3f}  "
              f"Compliance={compliance:.3f}  RCFS={score:.3f}")

rcfs_df = pd.DataFrame(rows).round(4)
print(rcfs_df.to_string(index=False))
rcfs_df.to_csv(os.path.join(TAB_DIR, "04_rcfs_by_technique.csv"), index=False)

# %%
# Which technique maximizes RCFS per dataset?
best = (rcfs_df.loc[rcfs_df.groupby("dataset")["RCFS"].idxmax()]
        [["dataset", "technique", "RCFS", "c_perf", "Compliance"]])
print("Best technique by RCFS:")
print(best.to_string(index=False))
best.to_csv(os.path.join(TAB_DIR, "04_rcfs_best_technique.csv"), index=False)

# %%
# Gamma sweep: trace the performance-compliance trade-off for each dataset,
# using the best-compliance technique's components.
sweep_rows = []
for (display_name, tech), comp in detail.items():
    for g in GAMMA_SWEEP:
        sweep_rows.append({
            "dataset": display_name, "technique": tech, "gamma": g,
            "RCFS": rcfs(comp["c_perf"], comp["compliance"], g),
        })
sweep_df = pd.DataFrame(sweep_rows).round(4)
# Pivot for readability: dataset x technique rows, gamma columns
sweep_pivot = sweep_df.pivot_table(index=["dataset", "technique"],
                                   columns="gamma", values="RCFS")
print(sweep_pivot.round(4).to_string())
sweep_df.to_csv(os.path.join(TAB_DIR, "04_rcfs_gamma_sweep.csv"), index=False)

[         German Credit] baseline       AUC=0.802  c_fair=0.985  c_expl=0.334  Compliance=0.499  RCFS=0.551
[         German Credit] reweighing     AUC=0.813  c_fair=0.976  c_expl=0.339  Compliance=0.503  RCFS=0.564
[         German Credit] reject_option  AUC=0.802  c_fair=0.997  c_expl=0.334  Compliance=0.500  RCFS=0.552
[     Taiwan CC Default] baseline       AUC=0.774  c_fair=0.926  c_expl=0.656  Compliance=0.768  RCFS=0.658
[     Taiwan CC Default] reweighing     AUC=0.775  c_fair=0.931  c_expl=0.653  Compliance=0.768  RCFS=0.659
[     Taiwan CC Default] reject_option  AUC=0.774  c_fair=1.000  c_expl=0.656  Compliance=0.792  RCFS=0.671
[  Credit Card Approval] baseline       AUC=0.690  c_fair=0.859  c_expl=0.477  Compliance=0.614  RCFS=0.497
[  Credit Card Approval] reweighing     AUC=0.692  c_fair=0.993  c_expl=0.472  Compliance=0.640  RCFS=0.512
[  Credit Card Approval] reject_option  AUC=0.690  c_fair=1.000  c_expl=0.477  Compliance=0.646  RCFS=0.513
             dataset     tec